<a href="https://colab.research.google.com/github/venkatasai-eng/MLA0305-REINFORCEMENT-LEARNING-/blob/main/EXP_NO_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



---



In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import random

np.random.seed(42)
tf.random.set_seed(42)

n_states = 5
n_actions = 2
episodes = 50
gamma = 0.9
learning_rate = 0.001
epsilon = 0.1

def environment(state, action):
    if action == 1:
        next_state = min(state + 1, 4)
    else:
        next_state = max(state - 1, 0)

    reward = 10 if next_state == 4 else -1
    done = next_state == 4

    return next_state, reward, done

def encode(state):
    x = np.zeros(n_states)
    x[state] = 1
    return x

def create_dqn():
    model = tf.keras.Sequential([
        layers.Input(shape=(n_states,)),
        layers.Dense(16, activation="relu"),
        layers.Dense(16, activation="relu"),
        layers.Dense(n_actions)
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="mse"
    )

    return model

def create_dueling_dqn():
    inputs = layers.Input(shape=(n_states,))
    x = layers.Dense(16, activation="relu")(inputs)
    x = layers.Dense(16, activation="relu")(x)

    value = layers.Dense(1)(x)
    advantage = layers.Dense(n_actions)(x)

    mean_advantage = layers.Lambda(
        lambda x: tf.reduce_mean(
            x, axis=1, keepdims=True
        )
    )(advantage)

    advantage_centered = layers.Subtract()([
        advantage,
        mean_advantage
    ])

    q_values = layers.Add()([
        value,
        advantage_centered
    ])

    model = tf.keras.Model(
        inputs=inputs,
        outputs=q_values
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="mse"
    )

    return model

def train(model):
    rewards = []

    for episode in range(episodes):

        state = 0
        total_reward = 0

        for _ in range(10):

            current_state = encode(state).reshape(1, -1)

            q_values = model.predict(
                current_state,
                verbose=0
            )[0]

            if random.random() < epsilon:
                action = random.randint(0, n_actions - 1)
            else:
                action = np.argmax(q_values)

            next_state, reward, done = environment(
                state,
                action
            )

            if done:
                target = reward
            else:
                next_state_data = encode(
                    next_state
                ).reshape(1, -1)

                next_q_values = model.predict(
                    next_state_data,
                    verbose=0
                )[0]

                target = reward + gamma * np.max(
                    next_q_values
                )

            q_values[action] = target

            model.fit(
                current_state,
                q_values.reshape(1, -1),
                epochs=1,
                verbose=0
            )

            total_reward += reward
            state = next_state

            if done:
                break

        rewards.append(total_reward)

    return rewards

dqn = create_dqn()
double_dqn = create_dqn()
dueling_dqn = create_dueling_dqn()

dqn_rewards = train(dqn)
double_rewards = train(double_dqn)
dueling_rewards = train(dueling_dqn)

print("DQN Average Reward:",
      round(np.mean(dqn_rewards[-10:]), 2))

print("Double DQN Average Reward:",
      round(np.mean(double_rewards[-10:]), 2))

print("Dueling DQN Average Reward:",
      round(np.mean(dueling_rewards[-10:]), 2))

print("\nDQN Q-Values:")
print(np.round(
    dqn.predict(
        np.eye(n_states),
        verbose=0
    ), 2
))

print("\nDouble DQN Q-Values:")
print(np.round(
    double_dqn.predict(
        np.eye(n_states),
        verbose=0
    ), 2
))

print("\nDueling DQN Q-Values:")
print(np.round(
    dueling_dqn.predict(
        np.eye(n_states),
        verbose=0
    ), 2
))

print("\nTraining completed successfully.")

DQN Average Reward: 6.3
Double DQN Average Reward: 6.9
Dueling DQN Average Reward: 7.0

DQN Q-Values:
[[-0.01 -0.  ]
 [-0.15 -0.15]
 [-0.3   0.23]
 [-1.43  1.71]
 [-0.55  0.43]]

Double DQN Q-Values:
[[-7.15 -4.47]
 [-7.05 -4.21]
 [-6.   -3.49]
 [-2.81 -0.12]
 [-6.56 -3.64]]

Dueling DQN Q-Values:
[[0.24 1.14]
 [0.04 0.4 ]
 [0.27 0.77]
 [0.65 3.14]
 [0.44 1.48]]

Training completed successfully.
